#Preparación entorno

In [41]:

from pathlib import Path
import sys
import pandas as pd

# Identificar la carpeta AE1 del proyecto
ruta_proyecto = Path.cwd()

if ruta_proyecto.name == "notebooks":
    ruta_proyecto = ruta_proyecto.parent
elif ruta_proyecto.name == "AD_2026Q4_Grupo20":
    ruta_proyecto = ruta_proyecto / "AE1"

if str(ruta_proyecto) not in sys.path:
    sys.path.insert(0, str(ruta_proyecto))

print("Carpeta de la actividad:", ruta_proyecto)
print("Versión de pandas:", pd.__version__)

Carpeta de la actividad: /workspaces/AD_2026Q4_Grupo20/AE1
Versión de pandas: 3.1.0


#Cargar dataset usando la clase del primer compañero

In [42]:
from src.cargador_dataset import HFDatasetLoader

# Crear el cargador y obtener los datos de Hugging Face
cargador = HFDatasetLoader(
    dataset_id="SIH/palmer-penguins",
    split="train"
)

df = cargador.cargar_datos()

print("Dataset cargado correctamente.")
print("Cantidad de filas y columnas:", df.shape)

display(df.head())

Dataset cargado correctamente.
Cantidad de filas y columnas: (344, 8)


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


#Explorar datos

In [43]:
print("Nombres de las columnas:")
print(df.columns.tolist())

print("\nTipos de datos:")
print(df.dtypes)

print("\nValores faltantes por columna:")
print(df.isnull().sum())

print("\nEstadísticas descriptivas:")
display(df.describe(include="all"))

Nombres de las columnas:
['species', 'island', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'sex', 'year']

Tipos de datos:
species                  str
island                   str
bill_length_mm       float64
bill_depth_mm        float64
flipper_length_mm    float64
body_mass_g          float64
sex                      str
year                   int64
dtype: object

Valores faltantes por columna:
species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
year                  0
dtype: int64

Estadísticas descriptivas:


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
count,344,344,342.000000,342.000000,342.000000,342.000000,333,344.000000
unique,3,3,NaN,NaN,NaN,NaN,2,NaN
top,Adelie,Biscoe,NaN,NaN,NaN,NaN,male,NaN
freq,152,168,NaN,NaN,NaN,NaN,168,NaN
mean,NaN,NaN,43.921930,17.151170,200.915205,4201.754386,NaN,2008.029070
std,NaN,NaN,5.459584,1.974793,14.061714,801.954536,NaN,0.818356
min,NaN,NaN,32.100000,13.100000,172.000000,2700.000000,NaN,2007.000000
25%,NaN,NaN,39.225000,15.600000,190.000000,3550.000000,NaN,2007.000000
50%,NaN,NaN,44.450000,17.300000,197.000000,4050.000000,NaN,2008.000000
75%,NaN,NaN,48.500000,18.700000,213.000000,4750.000000,NaN,2009.000000


# Transformación 1: tratar los valores faltantes

In [44]:
df_transformado = df.copy()

columnas_numericas = [
    "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "body_mass_g"
]

for columna in columnas_numericas:
    mediana = df_transformado[columna].median()
    df_transformado[columna] = (
        df_transformado[columna].fillna(mediana)
    )

df_transformado["sex"] = (
    df_transformado["sex"].fillna("unknown")
)

print("Valores faltantes después de la transformación:")
print(df_transformado.isnull().sum())

Valores faltantes después de la transformación:
species              0
island               0
bill_length_mm       0
bill_depth_mm        0
flipper_length_mm    0
body_mass_g          0
sex                  0
year                 0
dtype: int64


# Transformación 2: crear una nueva variable

In [45]:
df_transformado["bill_ratio"] = (
    df_transformado["bill_length_mm"]
    / df_transformado["bill_depth_mm"]
)

display(
    df_transformado[
        ["bill_length_mm", "bill_depth_mm", "bill_ratio"]
    ].head()
)

,bill_length_mm,bill_depth_mm,bill_ratio
0,39.10,18.7,2.090909
1,39.50,17.4,2.270115
2,40.30,18.0,2.238889
3,44.45,17.3,2.569364
4,36.70,19.3,1.901554


# Transformación 3: categorizar el peso

In [46]:
df_transformado["weight_category"] = pd.cut(
    df_transformado["body_mass_g"],
    bins=[0, 3500, 4500, float("inf")],
    labels=["Ligero", "Medio", "Pesado"],
    include_lowest=True
)

print("Cantidad de pingüinos por categoría de peso:")
print(df_transformado["weight_category"].value_counts())

display(
    df_transformado[
        ["species", "body_mass_g", "weight_category"]
    ].head()
)

Cantidad de pingüinos por categoría de peso:
weight_category
Medio     151
Pesado    115
Ligero     78
Name: count, dtype: int64


,species,body_mass_g,weight_category
0,Adelie,3750.0,Medio
1,Adelie,3800.0,Medio
2,Adelie,3250.0,Ligero
3,Adelie,4050.0,Medio
4,Adelie,3450.0,Ligero


# Transformación 4: filtrar por isla

In [47]:
df_biscoe = df_transformado[
    df_transformado["island"] == "Biscoe"
].copy()

print("Registros correspondientes a la isla Biscoe:")
print("Cantidad de filas:", len(df_biscoe))

display(df_biscoe.head())

Registros correspondientes a la isla Biscoe:
Cantidad de filas: 168


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year,bill_ratio,weight_category
20,Adelie,Biscoe,37.8,18.3,174.0,3400.0,female,2007,2.065574,Ligero
21,Adelie,Biscoe,37.7,18.7,180.0,3600.0,male,2007,2.016043,Medio
22,Adelie,Biscoe,35.9,19.2,189.0,3800.0,female,2007,1.869792,Medio
23,Adelie,Biscoe,38.2,18.1,185.0,3950.0,male,2007,2.110497,Medio
24,Adelie,Biscoe,38.8,17.2,180.0,3800.0,male,2007,2.255814,Medio


# Analizar el peso promedio por especie

In [48]:
resumen_especies = (
    df_transformado
    .groupby("species", observed=True)["body_mass_g"]
    .mean()
    .round(2)
    .reset_index(name="peso_promedio_g")
)

display(resumen_especies)

,species,peso_promedio_g
0,Adelie,3702.96
1,Chinstrap,3733.09
2,Gentoo,5067.74


# Generar el reporte con la clase del estudiante 2

In [49]:
from pathlib import Path
import sys

# Identificar la carpeta principal del proyecto
ruta_proyecto = Path.cwd()

if not (ruta_proyecto / "src").exists():
    ruta_proyecto = ruta_proyecto.parent

sys.path.insert(0, str(ruta_proyecto))

from src.resumidor_dataset import DatasetSummarizer

# Definir la ruta del reporte
ruta_reporte = ruta_proyecto / "reportes" / "resumen_dataset.txt"

# Generar el reporte con el DataFrame transformado
resumidor = DatasetSummarizer(
    df_transformado,
    output_path=str(ruta_reporte)
)

resumidor.generar_resumen()

print("\n¿Se creó el reporte?", ruta_reporte.exists())

print("\nContenido del reporte:")
print(ruta_reporte.read_text(encoding="utf-8"))

Resumen guardado en: /workspaces/AD_2026Q4_Grupo20/AE1/reportes/resumen_dataset.txt

¿Se creó el reporte? True

Contenido del reporte:
RESUMEN DEL DATASET

Cantidad de filas: 344
Cantidad de columnas: 10

NOMBRES DE LAS COLUMNAS
---------------------------------------------
- species
- island
- bill_length_mm
- bill_depth_mm
- flipper_length_mm
- body_mass_g
- sex
- year
- bill_ratio
- weight_category

TIPOS DE DATOS
---------------------------------------------
species                   str
island                    str
bill_length_mm        float64
bill_depth_mm         float64
flipper_length_mm     float64
body_mass_g           float64
sex                       str
year                    int64
bill_ratio            float64
weight_category      category

VALORES FALTANTES POR COLUMNA
---------------------------------------------
species              0
island               0
bill_length_mm       0
bill_depth_mm        0
flipper_length_mm    0
body_mass_g          0
sex               

# Validar los resultados

In [50]:
assert df.shape == (344, 8), "Las dimensiones originales no son las esperadas."

assert df_transformado[
    columnas_numericas + ["sex"]
].isnull().sum().sum() == 0, "Quedan valores faltantes en las columnas tratadas."

assert df_transformado["bill_ratio"].notna().all(), "Hay valores faltantes en bill_ratio."

assert df_transformado["weight_category"].notna().all(), "Hay registros sin categoría de peso."

assert ruta_reporte.exists(), "No se generó el archivo de reporte."

print("Todas las validaciones finalizaron correctamente.")

Todas las validaciones finalizaron correctamente.
